In [2]:
import os
import sys
import numpy as np
import warnings
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, ConstantKernel
from scipy.optimize import minimize
from scipy.stats import norm
from pathlib import Path
sys.path.insert(0, "../Utilities/")
import common_functions as cf
warnings.filterwarnings('ignore')

In [3]:
def ucb(x, gp, Y_max, beta):
    #Upper Confidence Bound (UCB) acquisition function
    x = x.reshape(1, -1)

    # Get the mean (mu) and std (sigma) from the GP
    mu, sigma = gp.predict(x, return_std=True)
    
    # Calculate UCB: mu + sqrt(beta) * sigma
    ucb = mu + np.sqrt(beta) * sigma
    
    return -ucb[0] # Return the negative UCB for minimisation

In [4]:
submission_queries = {} # Dictionary to hold all 8 query strings
root = Path.cwd().parent.parent

base_directory = '..\..\data\initial_data'
beta_exploration = 2.0 #will increase for functions 6 to 8

for i in range(1, 9):
    #print(f"\n--- Function {i} ---")
    
    folder_name = f'function_{i}'
    folder_path = os.path.join(root, "data/initial_data", folder_name)
    #print(os.path.join(folder_path, 'initial_inputs.npy'))
    X = np.load(os.path.join(folder_path, 'initial_inputs.npy'))
    Y = np.load(os.path.join(folder_path, 'initial_outputs.npy'))
    
    #print(X.shape)
    # dimensionality
    dimension = X.shape[1]
    
    if Y.ndim > 1:
        Y = Y.flatten()
        
    Y_max = np.max(Y)

    
    #print(f"{dimension}-Dimension - Y_max: {Y_max:.4f}")
    
    # 2. Build and Train GP
    kernel = Matern(length_scale=np.ones(dimension), length_scale_bounds=(1e-1, 1e2), nu=2.5)
    gp = GaussianProcessRegressor(
        kernel=kernel, alpha=1e-6, n_restarts_optimizer=20, random_state=42)
    gp.fit(X, Y)
    
    # 3. Define the Optimisation Task
    # Bounds for all functions are assumed to be [0.0, 1.0] for all dimensions.
    bounds = [(0.0, 1.0)] * dimension
    
    # Set the UCB exploration parameter based on dimensionality (Strategy: High Beta)
    # Increase Beta for higher dimensions where data is sparser.
    beta = beta_exploration
    if dimension >= 5:
        beta = beta_exploration * 1.5 # More aggressive exploration for high-D
        
    # The objective function (to be minimse) is the negative UCB
    ucb_objective = lambda x: ucb(x, gp, Y_max, beta)
    
    # 4. Find the Argmax (i.e., minimise the negative UCB)
    # Use multiple random starts to avoid local minima in the acquisition function
    best_ucb_value = np.inf
    
    x_next = None
    
    for _ in range(20*dimension): # 20*D random restarts for better robustness
        x0 = np.random.uniform(0, 1, dimension)
        
        # Use a general minimiser (L-BFGS-B is good for bounded problems)
        res = minimize(ucb_objective, x0, bounds=bounds, method='L-BFGS-B')
        
        if res.fun < best_ucb_value:
            best_ucb_value = res.fun
            x_next = res.x
    print(cf.format_inputdata(x_next))

0.000000-0.000000
1.000000-0.000000
1.000000-1.000000-0.837642
0.531712-0.402180-0.000000-0.275240
0.232877-0.841416-0.883342-0.879464
0.314365-0.000000-0.399314-1.000000-0.000000
0.000000-0.210370-1.000000-0.000000-0.323749-1.000000
0.141081-0.102602-0.176131-0.000000-1.000000-0.409759-0.147143-0.000000
